documentation - https://docs.databricks.com/aws/en/machine-learning/feature-store/
https://victorbnnt.medium.com/prepare-databricks-certified-ml-professional-certification-exam-21ccba833a5c

In [0]:
%pip install databricks-feature-store
dbutils.library.restartPython()

In [0]:
import pandas as pd
import seaborn as sns
from pyspark.sql.functions import *
from databricks.feature_store import FeatureStoreClient, feature_table

In [0]:
taxis_df = sns.load_dataset("taxis")
taxis_sdf = spark.createDataFrame(taxis_df)
display(taxis_sdf.limit(5))

In [0]:
(taxis_sdf.write
          .mode("overwrite")
          .option("overwriteSchema", "True")
          .format("delta")
          .saveAsTable("silver.default.taxis"))

In [0]:
taxis_df = spark.sql('select * from silver.default.taxis')
taxis_df = taxis_df.withColumn("id", monotonically_increasing_id())

In [0]:
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()

customer_feature_table = fs.create_table(
  name='silver.default.taxisfs',  
  primary_keys='id',                            # required
  schema=taxis_df.schema,                      # here only schema is provided, the feature table is created empty
  description='Seaborn taxi dataset features'
)

In [0]:
fs.write_table(
  df=taxis_df,
  name='silver.default.taxisfs',
  mode='merge'                   # mode = 'overwrite' could also be used in this particular case
)

In [0]:
display(fs.read_table(name='silver.default.taxisfs').limit(5))

In [0]:
%sql
DESCRIBE HISTORY silver.default.taxisfs